In [ ]:
!pip install tabpfn scikit-learn pandas numpy

In [ ]:
# ============================================================
# Clasificación con TabPFN
# Dataset: heart.csv
# ============================================================

import os

# Token TabPFN
os.environ["TABPFN_TOKEN"] = "tabpfn_sk_1sdkrTL19lqyiA9HCKDNRfVMoyHTUj9a3BVtW6cIPbk"

token = os.getenv("TABPFN_TOKEN")

print("Python ve el token:", token is not None and len(token) > 0)
print("Longitud token:", len(token) if token else 0)

# ============================================================
# Librerías
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# ============================================================
# Cargar datos
# ============================================================

heart = pd.read_csv("heart.csv")

X = heart.drop(columns=["target"])
y = heart["target"]

print("Shape X:", X.shape)
print("Shape y:", y.shape)

print(X.head())
print(y.head())

# Clases existentes
class_names = [str(c) for c in sorted(y.unique())]

print("Clases:", class_names)

# ============================================================
# Train / Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=123,
    stratify=y
)

# ============================================================
# Entrenamiento
# ============================================================

modelo_tabpfn = TabPFNClassifier.create_default_for_version(
    ModelVersion.V2
)

modelo_tabpfn.fit(X_train, y_train)

# ============================================================
# Predicciones
# ============================================================

y_pred = modelo_tabpfn.predict(X_test)

y_proba = modelo_tabpfn.predict_proba(X_test)

# ============================================================
# Resultados
# ============================================================

resultados = X_test.copy()

resultados["y_real"] = y_test.values
resultados["y_pred"] = y_pred

print(resultados.head())

# ============================================================
# Accuracy
# ============================================================

acc = accuracy_score(y_test, y_pred)

print("\nAccuracy:", acc)

# ============================================================
# Matriz de confusión
# ============================================================

cm = confusion_matrix(y_test, y_pred)

print("\nMatriz de confusión:")
print(cm)

# ============================================================
# Reporte
# ============================================================

print("\nReporte de clasificación:")

print(
    classification_report(
        y_test,
        y_pred
    )
)

# ============================================================
# Probabilidades
# ============================================================

proba_df = pd.DataFrame(
    y_proba,
    columns=[
        f"prob_clase_{c}"
        for c in modelo_tabpfn.classes_
    ]
)

print("\nProbabilidades:")
print(proba_df.head())

# ============================================================
# Ejemplo de nuevo paciente
# ============================================================

nuevo_caso = pd.DataFrame({
    "age": [63],
    "sex": [1],
    "cp": [3],
    "trestbps": [145],
    "chol": [233],
    "fbs": [1],
    "restecg": [0],
    "thalach": [150],
    "exang": [0],
    "oldpeak": [2.3],
    "slope": [0],
    "ca": [0],
    "thal": [1]
})

pred_nuevo = modelo_tabpfn.predict(nuevo_caso)
proba_nuevo = modelo_tabpfn.predict_proba(nuevo_caso)

print("\nPredicción nuevo caso:")
print("Clase:", pred_nuevo[0])

print(
    pd.DataFrame(
        proba_nuevo,
        columns=[
            f"prob_clase_{c}"
            for c in modelo_tabpfn.classes_
        ]
    )
)

Python ve el token: True
Longitud token: 53
Shape X: (1025, 13)
Shape y: (1025,)
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  
0   2     3  
1   0     3  
2   0     3  
3   1     3  
4   3     2  
0    0
1    0
2    0
3    0
4    0
Name: target, dtype: int64
Clases: ['0', '1']


/usr/local/lib/python3.12/dist-packages/tabpfn/validation.py:142: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(


     age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  \
341   44    1   2       120   226    0        1      169      0      0.0   
46    41    1   1       135   203    0        1      132      0      0.0   
152   58    1   0       125   300    0        0      171      0      0.0   
691   55    0   1       135   250    0        0      161      0      1.4   
288   58    0   2       120   340    0        1      172      0      0.0   

     slope  ca  thal  y_real  y_pred  
341      2   0     2       1       1  
46       1   0     1       1       1  
152      2   2     3       0       0  
691      1   0     2       1       1  
288      2   0     2       1       1  

Accuracy: 1.0

Matriz de confusión:
[[100   0]
 [  0 105]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       105

    accuracy                           1.00       205
   m

In [ ]:
heart.shape

(1025, 14)